# 03c Compare Live Vs History Baseline

Compare the latest live Gold V2 outputs against a historical Gold V2 baseline for the same Frankfurt cell scheme.


## Comparison Outputs

This notebook prepares three comparison views without creating new Delta tables:

1. `trend_compare_df`: live 15-minute trend against historical slot baseline
2. `horizontal_alerts_df`: live horizontal cell-windows ranked by deviation from historical baseline
3. `hotspot_compare_df`: live hotspot ranking versus historical hotspot ranking


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys
import yaml

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

with (repo_root / "configs" / "region_config.yaml").open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)
with (repo_root / "configs" / "pipeline_config.yaml").open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

def parse_int(raw_value: str, *, default: int) -> int:
    text = raw_value.strip()
    return default if not text else int(text)

def parse_float(raw_value: str, *, default: float) -> float:
    text = raw_value.strip()
    return default if not text else float(text)

def format_for_id(value) -> str:
    numeric = float(value)
    if numeric.is_integer():
        return str(int(numeric))
    return str(numeric).replace('.', 'p')

def build_default_cell_scheme_id(*, focus_airport: str, horizontal_cell_nm: float, vertical_cell_ft: float, analysis_window_minutes: int, min_altitude_ft: float, max_altitude_ft: float, projection_mode: str) -> str:
    return (
        f"{focus_airport.lower()}_h{format_for_id(horizontal_cell_nm)}nm_"
        f"v{format_for_id(vertical_cell_ft)}ft_"
        f"w{analysis_window_minutes}m_"
        f"a{format_for_id(min_altitude_ft)}to{format_for_id(max_altitude_ft)}_"
        f"{projection_mode}_v2"
    )

def latest_pipeline_run(*, catalog: str, pipeline_name: str, layer: str):
    return spark.sql(
        f"""
        SELECT run_id, metadata_json, completed_at
        FROM {catalog}.obs.pipeline_run_log
        WHERE pipeline_name = '{pipeline_name}'
          AND layer = '{layer}'
          AND status = 'success'
        ORDER BY completed_at DESC
        LIMIT 1
        """
    ).first()

def parse_cell_scheme_from_metadata(metadata_json: str | None):
    if not metadata_json:
        return None
    try:
        payload = json.loads(metadata_json)
    except Exception:
        return None
    return payload.get("cell_scheme_id")

def with_slot_columns(df, *, timestamp_column: str):
    return (
        df
        .withColumn("weekday_iso", F.expr(f"((dayofweek({timestamp_column}) + 5) % 7) + 1").cast("int"))
        .withColumn("window_minute_of_day", (F.hour(F.col(timestamp_column)) * F.lit(60) + F.minute(F.col(timestamp_column))).cast("int"))
    )

default_catalog = pipeline_config.get("catalog_name", "adsb_airspace_eddf")
default_live_run_id = ""
default_historical_run_id = ""
default_cell_scheme_id = ""
default_horizontal_cell_nm = "10"
default_vertical_cell_ft = "3000"
default_analysis_window_minutes = "15"
default_min_altitude_ft = "0"
default_max_altitude_ft = "24500"
default_projection_mode = "local_nm"
default_min_weekday_baseline_samples = "2"
default_top_n = "15"

catalog = default_catalog
live_run_id_raw = default_live_run_id
historical_run_id_raw = default_historical_run_id
cell_scheme_id_raw = default_cell_scheme_id
horizontal_cell_nm_raw = default_horizontal_cell_nm
vertical_cell_ft_raw = default_vertical_cell_ft
analysis_window_minutes_raw = default_analysis_window_minutes
min_altitude_ft_raw = default_min_altitude_ft
max_altitude_ft_raw = default_max_altitude_ft
projection_mode = default_projection_mode
min_weekday_baseline_samples_raw = default_min_weekday_baseline_samples
top_n_raw = default_top_n

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    dbutils.widgets.text("live_run_id", default_live_run_id)
    dbutils.widgets.text("historical_run_id", default_historical_run_id)
    dbutils.widgets.text("cell_scheme_id", default_cell_scheme_id)
    dbutils.widgets.text("horizontal_cell_nm", default_horizontal_cell_nm)
    dbutils.widgets.text("vertical_cell_ft", default_vertical_cell_ft)
    dbutils.widgets.text("analysis_window_minutes", default_analysis_window_minutes)
    dbutils.widgets.text("min_altitude_ft", default_min_altitude_ft)
    dbutils.widgets.text("max_altitude_ft", default_max_altitude_ft)
    dbutils.widgets.dropdown("projection_mode", default_projection_mode, ["local_nm"])
    dbutils.widgets.text("min_weekday_baseline_samples", default_min_weekday_baseline_samples)
    dbutils.widgets.text("top_n", default_top_n)

    catalog = dbutils.widgets.get("catalog").strip() or default_catalog
    live_run_id_raw = dbutils.widgets.get("live_run_id").strip()
    historical_run_id_raw = dbutils.widgets.get("historical_run_id").strip()
    cell_scheme_id_raw = dbutils.widgets.get("cell_scheme_id").strip()
    horizontal_cell_nm_raw = dbutils.widgets.get("horizontal_cell_nm").strip() or default_horizontal_cell_nm
    vertical_cell_ft_raw = dbutils.widgets.get("vertical_cell_ft").strip() or default_vertical_cell_ft
    analysis_window_minutes_raw = dbutils.widgets.get("analysis_window_minutes").strip() or default_analysis_window_minutes
    min_altitude_ft_raw = dbutils.widgets.get("min_altitude_ft").strip() or default_min_altitude_ft
    max_altitude_ft_raw = dbutils.widgets.get("max_altitude_ft").strip() or default_max_altitude_ft
    projection_mode = dbutils.widgets.get("projection_mode").strip() or default_projection_mode
    min_weekday_baseline_samples_raw = dbutils.widgets.get("min_weekday_baseline_samples").strip() or default_min_weekday_baseline_samples
    top_n_raw = dbutils.widgets.get("top_n").strip() or default_top_n

focus_airport = region_config["focus_airport"]
horizontal_cell_nm = parse_float(horizontal_cell_nm_raw, default=10.0)
vertical_cell_ft = parse_float(vertical_cell_ft_raw, default=3000.0)
analysis_window_minutes = parse_int(analysis_window_minutes_raw, default=15)
min_altitude_ft = parse_float(min_altitude_ft_raw, default=0.0)
max_altitude_ft = parse_float(max_altitude_ft_raw, default=24500.0)
min_weekday_baseline_samples = parse_int(min_weekday_baseline_samples_raw, default=2)
top_n = parse_int(top_n_raw, default=15)

if min_weekday_baseline_samples <= 0:
    raise ValueError("min_weekday_baseline_samples must be positive")
if top_n <= 0:
    raise ValueError("top_n must be positive")

default_cell_scheme_id = build_default_cell_scheme_id(
    focus_airport=focus_airport,
    horizontal_cell_nm=horizontal_cell_nm,
    vertical_cell_ft=vertical_cell_ft,
    analysis_window_minutes=analysis_window_minutes,
    min_altitude_ft=min_altitude_ft,
    max_altitude_ft=max_altitude_ft,
    projection_mode=projection_mode,
)

horizontal_complexity_table_v2 = f"{catalog}.gld_airspace.horizontal_complexity_v2"
horizontal_hotspots_table_v2 = f"{catalog}.gld_airspace.horizontal_hotspots_v2"
trend_table_v2 = f"{catalog}.gld_airspace.complexity_trend_v2"

latest_live_run = latest_pipeline_run(catalog=catalog, pipeline_name="03b_build_live_complexity_metrics", layer="gold_live_v2")
latest_historical_run = latest_pipeline_run(catalog=catalog, pipeline_name="03_build_complexity_metrics", layer="gold_v2")

if latest_live_run is None:
    raise ValueError("No successful live Gold run found. Run 03b_build_live_complexity_metrics.ipynb first.")
if latest_historical_run is None:
    raise ValueError("No successful historical Gold run found. Run 03_build_complexity_metrics.ipynb first.")

selected_live_run_id = live_run_id_raw or latest_live_run["run_id"]
selected_historical_run_id = historical_run_id_raw or latest_historical_run["run_id"]
selected_cell_scheme_id = cell_scheme_id_raw or parse_cell_scheme_from_metadata(latest_live_run["metadata_json"]) or default_cell_scheme_id

live_horizontal_df_v2 = (
    spark.table(horizontal_complexity_table_v2)
    .where(F.col("run_id") == F.lit(selected_live_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
live_hotspots_df_v2 = (
    spark.table(horizontal_hotspots_table_v2)
    .where(F.col("run_id") == F.lit(selected_live_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
live_trend_df_v2 = (
    spark.table(trend_table_v2)
    .where(F.col("run_id") == F.lit(selected_live_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)

historical_horizontal_df_v2 = (
    spark.table(horizontal_complexity_table_v2)
    .where(F.col("run_id") == F.lit(selected_historical_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
historical_hotspots_df_v2 = (
    spark.table(horizontal_hotspots_table_v2)
    .where(F.col("run_id") == F.lit(selected_historical_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)
historical_trend_df_v2 = (
    spark.table(trend_table_v2)
    .where(F.col("run_id") == F.lit(selected_historical_run_id))
    .where(F.col("cell_scheme_id") == F.lit(selected_cell_scheme_id))
)

live_trend_count = live_trend_df_v2.count()
historical_trend_count = historical_trend_df_v2.count()
live_horizontal_count = live_horizontal_df_v2.count()
historical_horizontal_count = historical_horizontal_df_v2.count()
live_hotspot_count = live_hotspots_df_v2.count()
historical_hotspot_count = historical_hotspots_df_v2.count()

if live_trend_count == 0 or historical_trend_count == 0:
    raise ValueError("Trend comparison requires both a live Gold run and a historical Gold run for the selected cell scheme.")
if live_horizontal_count == 0 or historical_horizontal_count == 0:
    raise ValueError("Horizontal comparison requires both a live Gold run and a historical Gold run for the selected cell scheme.")
if live_hotspot_count == 0 or historical_hotspot_count == 0:
    raise ValueError("Hotspot comparison requires both a live Gold run and a historical Gold run for the selected cell scheme.")

run_plan = {
    "catalog": catalog,
    "live_run_id": selected_live_run_id,
    "historical_run_id": selected_historical_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "min_weekday_baseline_samples": min_weekday_baseline_samples,
    "top_n": top_n,
    "live_trend_count": int(live_trend_count),
    "historical_trend_count": int(historical_trend_count),
    "live_horizontal_count": int(live_horizontal_count),
    "historical_horizontal_count": int(historical_horizontal_count),
    "live_hotspot_count": int(live_hotspot_count),
    "historical_hotspot_count": int(historical_hotspot_count),
}

run_plan


In [ ]:
live_trend_slots_df = with_slot_columns(live_trend_df_v2, timestamp_column="window_start")
historical_trend_slots_df = with_slot_columns(historical_trend_df_v2, timestamp_column="window_start")

historical_trend_weekday_baseline_df = (
    historical_trend_slots_df
    .groupBy("weekday_iso", "window_minute_of_day")
    .agg(
        F.count("*").alias("weekday_sample_count"),
        F.expr("percentile_approx(avg_complexity_score, 0.5, 10000)").alias("weekday_avg_complexity_p50"),
        F.expr("percentile_approx(avg_complexity_score, 0.9, 10000)").alias("weekday_avg_complexity_p90"),
        F.expr("percentile_approx(peak_complexity_score, 0.5, 10000)").alias("weekday_peak_complexity_p50"),
        F.expr("percentile_approx(peak_complexity_score, 0.9, 10000)").alias("weekday_peak_complexity_p90"),
        F.avg("active_horizontal_cell_count").alias("weekday_active_horizontal_cell_mean"),
        F.avg("active_aircraft_count").alias("weekday_active_aircraft_mean")
    )
)
historical_trend_anyday_baseline_df = (
    historical_trend_slots_df
    .groupBy("window_minute_of_day")
    .agg(
        F.count("*").alias("anyday_sample_count"),
        F.expr("percentile_approx(avg_complexity_score, 0.5, 10000)").alias("anyday_avg_complexity_p50"),
        F.expr("percentile_approx(avg_complexity_score, 0.9, 10000)").alias("anyday_avg_complexity_p90"),
        F.expr("percentile_approx(peak_complexity_score, 0.5, 10000)").alias("anyday_peak_complexity_p50"),
        F.expr("percentile_approx(peak_complexity_score, 0.9, 10000)").alias("anyday_peak_complexity_p90"),
        F.avg("active_horizontal_cell_count").alias("anyday_active_horizontal_cell_mean"),
        F.avg("active_aircraft_count").alias("anyday_active_aircraft_mean")
    )
)

trend_compare_df = (
    live_trend_slots_df.alias("live")
    .join(historical_trend_weekday_baseline_df.alias("weekday"), on=["weekday_iso", "window_minute_of_day"], how="left")
    .join(historical_trend_anyday_baseline_df.alias("anyday"), on=["window_minute_of_day"], how="left")
    .withColumn("use_weekday_baseline", F.coalesce(F.col("weekday.weekday_sample_count"), F.lit(0)) >= F.lit(min_weekday_baseline_samples))
    .withColumn("baseline_mode", F.when(F.col("use_weekday_baseline"), F.lit("weekday_slot")).otherwise(F.lit("slot_all_days")))
    .withColumn("baseline_sample_count", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_sample_count")).otherwise(F.col("anyday.anyday_sample_count")))
    .withColumn("baseline_avg_complexity_p50", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_avg_complexity_p50")).otherwise(F.col("anyday.anyday_avg_complexity_p50")))
    .withColumn("baseline_avg_complexity_p90", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_avg_complexity_p90")).otherwise(F.col("anyday.anyday_avg_complexity_p90")))
    .withColumn("baseline_peak_complexity_p50", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_peak_complexity_p50")).otherwise(F.col("anyday.anyday_peak_complexity_p50")))
    .withColumn("baseline_peak_complexity_p90", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_peak_complexity_p90")).otherwise(F.col("anyday.anyday_peak_complexity_p90")))
    .withColumn("baseline_active_horizontal_cell_mean", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_active_horizontal_cell_mean")).otherwise(F.col("anyday.anyday_active_horizontal_cell_mean")))
    .withColumn("baseline_active_aircraft_mean", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_active_aircraft_mean")).otherwise(F.col("anyday.anyday_active_aircraft_mean")))
    .withColumn("avg_complexity_delta_from_p50", F.col("avg_complexity_score") - F.col("baseline_avg_complexity_p50"))
    .withColumn("peak_complexity_delta_from_p90", F.col("peak_complexity_score") - F.col("baseline_peak_complexity_p90"))
    .withColumn("avg_complexity_above_p90_flag", F.when(F.col("baseline_avg_complexity_p90").isNull(), F.lit(False)).otherwise(F.col("avg_complexity_score") >= F.col("baseline_avg_complexity_p90")))
    .withColumn("peak_complexity_above_p90_flag", F.when(F.col("baseline_peak_complexity_p90").isNull(), F.lit(False)).otherwise(F.col("peak_complexity_score") >= F.col("baseline_peak_complexity_p90")))
    .select(
        F.col("live.window_start").alias("window_start"),
        F.col("weekday_iso"),
        F.col("window_minute_of_day"),
        F.col("live.avg_complexity_score").alias("live_avg_complexity_score"),
        F.col("live.peak_complexity_score").alias("live_peak_complexity_score"),
        F.col("live.active_horizontal_cell_count").alias("live_active_horizontal_cell_count"),
        F.col("live.active_aircraft_count").alias("live_active_aircraft_count"),
        F.col("baseline_mode"),
        F.col("baseline_sample_count"),
        F.round(F.col("baseline_avg_complexity_p50"), 6).alias("baseline_avg_complexity_p50"),
        F.round(F.col("baseline_avg_complexity_p90"), 6).alias("baseline_avg_complexity_p90"),
        F.round(F.col("baseline_peak_complexity_p50"), 6).alias("baseline_peak_complexity_p50"),
        F.round(F.col("baseline_peak_complexity_p90"), 6).alias("baseline_peak_complexity_p90"),
        F.round(F.col("baseline_active_horizontal_cell_mean"), 3).alias("baseline_active_horizontal_cell_mean"),
        F.round(F.col("baseline_active_aircraft_mean"), 3).alias("baseline_active_aircraft_mean"),
        F.round(F.col("avg_complexity_delta_from_p50"), 6).alias("avg_complexity_delta_from_p50"),
        F.round(F.col("peak_complexity_delta_from_p90"), 6).alias("peak_complexity_delta_from_p90"),
        F.col("avg_complexity_above_p90_flag"),
        F.col("peak_complexity_above_p90_flag"),
        F.lit(selected_live_run_id).alias("live_run_id"),
        F.lit(selected_historical_run_id).alias("historical_run_id"),
        F.lit(selected_cell_scheme_id).alias("cell_scheme_id")
    )
)

live_horizontal_slots_df = with_slot_columns(live_horizontal_df_v2, timestamp_column="window_start")
historical_horizontal_slots_df = with_slot_columns(historical_horizontal_df_v2, timestamp_column="window_start")

historical_horizontal_weekday_baseline_df = (
    historical_horizontal_slots_df
    .groupBy("weekday_iso", "window_minute_of_day", "horizontal_cell_id")
    .agg(
        F.count("*").alias("weekday_sample_count"),
        F.expr("percentile_approx(complexity_score, 0.5, 10000)").alias("weekday_complexity_p50"),
        F.expr("percentile_approx(complexity_score, 0.9, 10000)").alias("weekday_complexity_p90"),
        F.avg("aircraft_count").alias("weekday_aircraft_count_mean")
    )
)
historical_horizontal_anyday_baseline_df = (
    historical_horizontal_slots_df
    .groupBy("window_minute_of_day", "horizontal_cell_id")
    .agg(
        F.count("*").alias("anyday_sample_count"),
        F.expr("percentile_approx(complexity_score, 0.5, 10000)").alias("anyday_complexity_p50"),
        F.expr("percentile_approx(complexity_score, 0.9, 10000)").alias("anyday_complexity_p90"),
        F.avg("aircraft_count").alias("anyday_aircraft_count_mean")
    )
)

horizontal_alerts_df = (
    live_horizontal_slots_df.alias("live")
    .join(historical_horizontal_weekday_baseline_df.alias("weekday"), on=["weekday_iso", "window_minute_of_day", "horizontal_cell_id"], how="left")
    .join(historical_horizontal_anyday_baseline_df.alias("anyday"), on=["window_minute_of_day", "horizontal_cell_id"], how="left")
    .withColumn("use_weekday_baseline", F.coalesce(F.col("weekday.weekday_sample_count"), F.lit(0)) >= F.lit(min_weekday_baseline_samples))
    .withColumn("baseline_mode", F.when(F.col("use_weekday_baseline"), F.lit("weekday_slot_cell")).otherwise(F.lit("slot_cell_all_days")))
    .withColumn("baseline_sample_count", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_sample_count")).otherwise(F.col("anyday.anyday_sample_count")))
    .withColumn("baseline_complexity_p50", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_complexity_p50")).otherwise(F.col("anyday.anyday_complexity_p50")))
    .withColumn("baseline_complexity_p90", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_complexity_p90")).otherwise(F.col("anyday.anyday_complexity_p90")))
    .withColumn("baseline_aircraft_count_mean", F.when(F.col("use_weekday_baseline"), F.col("weekday.weekday_aircraft_count_mean")).otherwise(F.col("anyday.anyday_aircraft_count_mean")))
    .withColumn("complexity_delta_from_p50", F.col("live.complexity_score") - F.col("baseline_complexity_p50"))
    .withColumn("complexity_above_p90_flag", F.when(F.col("baseline_complexity_p90").isNull(), F.lit(False)).otherwise(F.col("live.complexity_score") >= F.col("baseline_complexity_p90")))
    .withColumn("missing_historical_baseline_flag", F.col("baseline_sample_count").isNull())
    .select(
        F.col("live.window_start").alias("window_start"),
        F.col("horizontal_cell_id"),
        F.col("weekday_iso"),
        F.col("window_minute_of_day"),
        F.col("live.aircraft_count").alias("live_aircraft_count"),
        F.col("live.active_vertical_cells").alias("live_active_vertical_cells"),
        F.round(F.col("live.complexity_score"), 6).alias("live_complexity_score"),
        F.col("baseline_mode"),
        F.col("baseline_sample_count"),
        F.round(F.col("baseline_complexity_p50"), 6).alias("baseline_complexity_p50"),
        F.round(F.col("baseline_complexity_p90"), 6).alias("baseline_complexity_p90"),
        F.round(F.col("baseline_aircraft_count_mean"), 3).alias("baseline_aircraft_count_mean"),
        F.round(F.col("complexity_delta_from_p50"), 6).alias("complexity_delta_from_p50"),
        F.col("complexity_above_p90_flag"),
        F.col("missing_historical_baseline_flag"),
        F.lit(selected_live_run_id).alias("live_run_id"),
        F.lit(selected_historical_run_id).alias("historical_run_id"),
        F.lit(selected_cell_scheme_id).alias("cell_scheme_id")
    )
)

hotspot_compare_df = (
    live_hotspots_df_v2.alias("live")
    .join(
        historical_hotspots_df_v2.select(
            F.col("horizontal_cell_id"),
            F.col("active_windows").alias("historical_active_windows"),
            F.col("avg_complexity_score").alias("historical_avg_complexity_score"),
            F.col("peak_complexity_score").alias("historical_peak_complexity_score"),
            F.col("ranking").alias("historical_ranking")
        ).alias("hist"),
        on="horizontal_cell_id",
        how="left"
    )
    .withColumn("new_hotspot_flag", F.col("historical_ranking").isNull())
    .withColumn("avg_complexity_score_delta", F.col("live.avg_complexity_score") - F.col("historical_avg_complexity_score"))
    .withColumn("peak_complexity_score_delta", F.col("live.peak_complexity_score") - F.col("historical_peak_complexity_score"))
    .withColumn("ranking_delta", F.when(F.col("historical_ranking").isNull(), F.lit(None)).otherwise(F.col("historical_ranking") - F.col("live.ranking")))
    .select(
        F.col("horizontal_cell_id"),
        F.col("live.ranking").alias("live_ranking"),
        F.col("historical_ranking"),
        F.col("live.active_windows").alias("live_active_windows"),
        F.col("historical_active_windows"),
        F.round(F.col("live.avg_complexity_score"), 6).alias("live_avg_complexity_score"),
        F.round(F.col("historical_avg_complexity_score"), 6).alias("historical_avg_complexity_score"),
        F.round(F.col("avg_complexity_score_delta"), 6).alias("avg_complexity_score_delta"),
        F.round(F.col("live.peak_complexity_score"), 6).alias("live_peak_complexity_score"),
        F.round(F.col("historical_peak_complexity_score"), 6).alias("historical_peak_complexity_score"),
        F.round(F.col("peak_complexity_score_delta"), 6).alias("peak_complexity_score_delta"),
        F.col("ranking_delta"),
        F.col("new_hotspot_flag"),
        F.lit(selected_live_run_id).alias("live_run_id"),
        F.lit(selected_historical_run_id).alias("historical_run_id"),
        F.lit(selected_cell_scheme_id).alias("cell_scheme_id")
    )
)

trend_summary_row = trend_compare_df.agg(
    F.count("*").alias("trend_rows"),
    F.sum(F.when(F.col("avg_complexity_above_p90_flag"), F.lit(1)).otherwise(F.lit(0))).cast("int").alias("avg_windows_above_p90"),
    F.sum(F.when(F.col("peak_complexity_above_p90_flag"), F.lit(1)).otherwise(F.lit(0))).cast("int").alias("peak_windows_above_p90")
).first()
horizontal_summary_row = horizontal_alerts_df.agg(
    F.count("*").alias("horizontal_alert_rows"),
    F.sum(F.when(F.col("complexity_above_p90_flag"), F.lit(1)).otherwise(F.lit(0))).cast("int").alias("cell_windows_above_p90"),
    F.sum(F.when(F.col("missing_historical_baseline_flag"), F.lit(1)).otherwise(F.lit(0))).cast("int").alias("cell_windows_missing_history")
).first()
hotspot_summary_row = hotspot_compare_df.agg(
    F.count("*").alias("hotspot_rows"),
    F.sum(F.when(F.col("new_hotspot_flag"), F.lit(1)).otherwise(F.lit(0))).cast("int").alias("new_hotspot_count")
).first()

comparison_summary = {
    "live_run_id": selected_live_run_id,
    "historical_run_id": selected_historical_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "trend_rows": int(trend_summary_row["trend_rows"]),
    "avg_windows_above_p90": int(trend_summary_row["avg_windows_above_p90"]),
    "peak_windows_above_p90": int(trend_summary_row["peak_windows_above_p90"]),
    "horizontal_alert_rows": int(horizontal_summary_row["horizontal_alert_rows"]),
    "cell_windows_above_p90": int(horizontal_summary_row["cell_windows_above_p90"]),
    "cell_windows_missing_history": int(horizontal_summary_row["cell_windows_missing_history"]),
    "hotspot_rows": int(hotspot_summary_row["hotspot_rows"]),
    "new_hotspot_count": int(hotspot_summary_row["new_hotspot_count"]),
}

comparison_summary


In [ ]:
trend_plot_df = trend_compare_df.orderBy("window_start").toPandas()
alerts_plot_df = (
    horizontal_alerts_df
    .orderBy(F.col("complexity_delta_from_p50").desc(), F.col("live_complexity_score").desc(), F.col("horizontal_cell_id").asc())
    .limit(top_n)
    .toPandas()
)
hotspot_plot_df = hotspot_compare_df.orderBy("live_ranking").limit(top_n).toPandas()

if trend_plot_df.empty:
    raise ValueError("trend_compare_df is empty; there is nothing to plot.")

trend_plot_df["window_start"] = pd.to_datetime(trend_plot_df["window_start"])
plt.style.use("ggplot")
fig, axes = plt.subplots(2, 1, figsize=(12, 10), constrained_layout=True)

axes[0].plot(trend_plot_df["window_start"], trend_plot_df["live_avg_complexity_score"], marker="o", linewidth=2, label="Live Avg")
axes[0].plot(trend_plot_df["window_start"], trend_plot_df["baseline_avg_complexity_p50"], marker="s", linewidth=1.8, label="Historical P50")
axes[0].plot(trend_plot_df["window_start"], trend_plot_df["baseline_avg_complexity_p90"], marker="^", linewidth=1.8, label="Historical P90")
axes[0].set_title("Live Trend Vs Historical Baseline")
axes[0].set_xlabel("Window Start")
axes[0].set_ylabel("Average Complexity Score")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend(loc="best")

if not alerts_plot_df.empty:
    alerts_plot_df = alerts_plot_df.sort_values("complexity_delta_from_p50", ascending=True)
    axes[1].barh(alerts_plot_df["horizontal_cell_id"], alerts_plot_df["complexity_delta_from_p50"], color="#d95f02")
    axes[1].set_title(f"Top {min(top_n, len(alerts_plot_df))} Horizontal Alerts Vs Baseline P50")
    axes[1].set_xlabel("Live Complexity - Historical P50")
    axes[1].set_ylabel("Horizontal Cell")
else:
    axes[1].text(0.5, 0.5, "No horizontal alerts found", ha="center", va="center")
    axes[1].set_axis_off()

plt.show()

if "display" in globals():
    display(trend_compare_df.orderBy("window_start"))
    display(horizontal_alerts_df.orderBy(F.col("complexity_delta_from_p50").desc(), F.col("live_complexity_score").desc()).limit(top_n))
    display(hotspot_compare_df.orderBy("live_ranking").limit(top_n))
else:
    print(trend_plot_df)
    print(alerts_plot_df)
    print(hotspot_plot_df)

final_summary = {
    "status": "success",
    "live_run_id": selected_live_run_id,
    "historical_run_id": selected_historical_run_id,
    "cell_scheme_id": selected_cell_scheme_id,
    "trend_rows": comparison_summary["trend_rows"],
    "horizontal_alert_rows": comparison_summary["horizontal_alert_rows"],
    "hotspot_rows": comparison_summary["hotspot_rows"],
    "avg_windows_above_p90": comparison_summary["avg_windows_above_p90"],
    "peak_windows_above_p90": comparison_summary["peak_windows_above_p90"],
    "new_hotspot_count": comparison_summary["new_hotspot_count"],
}

final_summary
